# Connecting Layers: Trace Layer 3 → Layer 2 → Original Data

This notebook traces the hierarchical clustering back to original indices:
- **Layer 3**: 10 clusters (cluster_second_round/)
- **Layer 2**: 80 clusters (cluster_first_round/)
- **Layer 1**: Original failure cases with original_index and current_index

In [1]:
import os
import json
import pandas as pd
from collections import defaultdict

## Configuration

In [2]:
# Directory paths
LAYER3_DIR = "./cluster_second_round"  # 10 clusters
LAYER2_DIR = "./cluster_first_round"   # 80 clusters
LAYER3_ANALYSIS_DIR = "results/cluster_analysis_layer3"
LAYER2_ANALYSIS_DIR = "results/cluster_analysis_layer2"
OUTPUT_DIR = "results/connecting_layers"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Output directory: {OUTPUT_DIR}")

✓ Output directory: results/connecting_layers


## Load Layer 2 Data (80 clusters with original items)

In [3]:
import glob

# Load all 80 layer 2 clusters
layer2_clusters = {}
layer2_analysis = {}

# Load raw cluster data (with original_index, current_index, failed_summary)
layer2_files = sorted(glob.glob(os.path.join(LAYER2_DIR, "cluster_*.json")),
                     key=lambda x: int(os.path.basename(x).replace('cluster_', '').replace('.json', '')))

for file_path in layer2_files:
    cluster_label = int(os.path.basename(file_path).replace('cluster_', '').replace('.json', ''))
    with open(file_path, 'r', encoding='utf-8') as f:
        layer2_clusters[cluster_label] = json.load(f)

# Load layer 2 analysis (with cluster_summary)
layer2_analysis_files = sorted(glob.glob(os.path.join(LAYER2_ANALYSIS_DIR, "cluster_*_analysis.json")),
                               key=lambda x: int(os.path.basename(x).replace('cluster_', '').replace('_analysis.json', '')))

for file_path in layer2_analysis_files:
    cluster_label = int(os.path.basename(file_path).replace('cluster_', '').replace('_analysis.json', ''))
    with open(file_path, 'r', encoding='utf-8') as f:
        layer2_analysis[cluster_label] = json.load(f)

print(f"✓ Loaded {len(layer2_clusters)} layer 2 raw clusters")
print(f"✓ Loaded {len(layer2_analysis)} layer 2 analysis files")

# Show structure of one layer 2 cluster
if 0 in layer2_clusters:
    sample_item = layer2_clusters[0][0]
    print(f"\nSample layer 2 item fields: {list(sample_item.keys())}")
    print(f"  - original_index: {sample_item.get('original_index')}")
    print(f"  - current_index: {sample_item.get('current_index')}")
    print(f"  - failed_summary: {sample_item.get('failed_summary')[:100]}...")

✓ Loaded 80 layer 2 raw clusters
✓ Loaded 80 layer 2 analysis files

Sample layer 2 item fields: ['cluster_label', 'original_index', 'current_index', 'failed_summary', 'embedding_for_clustering']
  - original_index: 1091
  - current_index: 353
  - failed_summary: The model likely focused on the surface facts (Li Hua’s team won first place and he personally won a...


## Load Layer 3 Data (10 clusters)

In [4]:
# Load all 10 layer 3 clusters
layer3_clusters = {}
layer3_analysis = {}

# Load raw cluster data (with round1_cluster_label)
layer3_files = sorted(glob.glob(os.path.join(LAYER3_DIR, "round2cluster_*.json")),
                     key=lambda x: int(os.path.basename(x).replace('round2cluster_', '').replace('.json', '')))

for file_path in layer3_files:
    cluster_label = int(os.path.basename(file_path).replace('round2cluster_', '').replace('.json', ''))
    with open(file_path, 'r', encoding='utf-8') as f:
        layer3_clusters[cluster_label] = json.load(f)

# Load layer 3 analysis (with cluster_summary)
layer3_analysis_files = sorted(glob.glob(os.path.join(LAYER3_ANALYSIS_DIR, "cluster_*_analysis.json")),
                               key=lambda x: int(os.path.basename(x).replace('cluster_', '').replace('_analysis.json', '')))

for file_path in layer3_analysis_files:
    cluster_label = int(os.path.basename(file_path).replace('cluster_', '').replace('_analysis.json', ''))
    with open(file_path, 'r', encoding='utf-8') as f:
        layer3_analysis[cluster_label] = json.load(f)

print(f"✓ Loaded {len(layer3_clusters)} layer 3 raw clusters")
print(f"✓ Loaded {len(layer3_analysis)} layer 3 analysis files")

# Show structure of one layer 3 cluster
if 0 in layer3_clusters:
    sample_item = layer3_clusters[0][0]
    print(f"\nSample layer 3 item fields: {list(sample_item.keys())}")
    print(f"  - round1_cluster_label: {sample_item.get('round1_cluster_label')}")
    print(f"  - cluster_summary: {sample_item.get('cluster_summary')[:100]}...")

✓ Loaded 10 layer 3 raw clusters
✓ Loaded 10 layer 3 analysis files

Sample layer 3 item fields: ['cluster_label', 'round1_cluster_label', 'cluster_summary', 'embedding_for_clustering']
  - round1_cluster_label: 29
  - cluster_summary: Across these cases, the model leans on stereotypical event–emotion pairings from surface cues (“revo...


## Connect Layers: Layer 3 → Layer 2 → Original Data

In [5]:
# Build complete hierarchical structure
complete_hierarchy = []

for layer3_label in sorted(layer3_clusters.keys()):
    layer3_items = layer3_clusters[layer3_label]
    layer3_summary = layer3_analysis.get(layer3_label, {}).get('cluster_summary', 'N/A')
    
    # Get all layer 2 clusters that belong to this layer 3 cluster
    round1_cluster_labels = [item['round1_cluster_label'] for item in layer3_items]
    
    # For each layer 2 cluster, get all original items
    layer2_details = []
    all_original_indices = []
    all_current_indices = []
    
    for layer2_label in round1_cluster_labels:
        layer2_raw_items = layer2_clusters.get(layer2_label, [])
        layer2_summary = layer2_analysis.get(layer2_label, {}).get('cluster_summary', 'N/A')
        
        # Extract original data for each item in this layer 2 cluster
        original_items = []
        for item in layer2_raw_items:
            original_items.append({
                'original_index': item.get('original_index'),
                'current_index': item.get('current_index'),
                'failed_summary': item.get('failed_summary')
            })
            all_original_indices.append(item.get('original_index'))
            all_current_indices.append(item.get('current_index'))
        
        layer2_details.append({
            'layer2_cluster_label': layer2_label,
            'layer2_cluster_summary': layer2_summary,
            'layer2_cluster_size': len(original_items),
            'original_items': original_items
        })
    
    complete_hierarchy.append({
        'layer3_cluster_label': layer3_label,
        'layer3_cluster_summary': layer3_summary,
        'layer3_cluster_size': len(round1_cluster_labels),
        'total_original_items': len(all_original_indices),
        'all_original_indices': sorted(all_original_indices),
        'all_current_indices': sorted(all_current_indices),
        'layer2_clusters': layer2_details
    })

print(f"✓ Built complete hierarchy for {len(complete_hierarchy)} layer 3 clusters")
print(f"\nExample structure for layer 3 cluster 0:")
print(f"  - Layer 3 summary: {complete_hierarchy[0]['layer3_cluster_summary'][:100]}...")
print(f"  - Contains {complete_hierarchy[0]['layer3_cluster_size']} layer 2 clusters")
print(f"  - Total original items: {complete_hierarchy[0]['total_original_items']}")
print(f"  - First few original indices: {complete_hierarchy[0]['all_original_indices'][:10]}")

✓ Built complete hierarchy for 10 layer 3 clusters

Example structure for layer 3 cluster 0:
  - Layer 3 summary: Both reasons describe the model over‑relying on stereotypical associations between surface events an...
  - Contains 2 layer 2 clusters
  - Total original items: 17
  - First few original indices: [95, 164, 488, 584, 596, 680, 716, 754, 873, 903]


## Save Complete Hierarchy

In [6]:
# Save individual files for each layer 3 cluster
for cluster_data in complete_hierarchy:
    layer3_label = cluster_data['layer3_cluster_label']
    output_file = os.path.join(OUTPUT_DIR, f"layer3_cluster_{layer3_label}_complete.json")
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(cluster_data, f, indent=2, ensure_ascii=False)

print(f"✓ Saved {len(complete_hierarchy)} individual hierarchy files to {OUTPUT_DIR}")
print(f"  Files: layer3_cluster_0_complete.json, layer3_cluster_1_complete.json, ...")

✓ Saved 10 individual hierarchy files to results/connecting_layers
  Files: layer3_cluster_0_complete.json, layer3_cluster_1_complete.json, ...


In [7]:
# Save a summary CSV for quick overview
summary_data = []
for cluster_data in complete_hierarchy:
    summary_data.append({
        'layer3_cluster_label': cluster_data['layer3_cluster_label'],
        'layer3_summary': cluster_data['layer3_cluster_summary'],
        'num_layer2_clusters': cluster_data['layer3_cluster_size'],
        'total_original_items': cluster_data['total_original_items'],
        'original_indices_sample': str(cluster_data['all_original_indices'][:20]),
        'layer2_cluster_labels': str([l2['layer2_cluster_label'] for l2 in cluster_data['layer2_clusters']])
    })

summary_df = pd.DataFrame(summary_data)
summary_csv = os.path.join(OUTPUT_DIR, "hierarchy_summary.csv")
summary_df.to_csv(summary_csv, index=False)

print(f"✓ Saved summary CSV to {summary_csv}")
print(f"\nSummary:")
print(summary_df)

✓ Saved summary CSV to results/connecting_layers/hierarchy_summary.csv

Summary:
   layer3_cluster_label                                     layer3_summary  \
0                     0  Both reasons describe the model over‑relying o...   
1                     1  Across these reasons, the model resolves under...   
2                     2  Across the cluster, the model over-relies on l...   
3                     3  Taken together, these reasons portray a model ...   
4                     4  These reasons all describe a shared failure in...   
5                     5  Across the cluster, the model applies a rigid,...   
6                     6  Across the cluster, the model does shallow pat...   
7                     7  Across the reasons, the model is portrayed as ...   
8                     8  Across this cluster, the model repeatedly igno...   
9                     9  Across this cluster, the model’s answers syste...   

   num_layer2_clusters  total_original_items  \
0           

## Create Index Lookup Tables

In [8]:
# Create lookup: original_index -> layer 3 cluster, layer 2 cluster
original_index_lookup = {}
current_index_lookup = {}

for cluster_data in complete_hierarchy:
    layer3_label = cluster_data['layer3_cluster_label']
    
    for layer2_data in cluster_data['layer2_clusters']:
        layer2_label = layer2_data['layer2_cluster_label']
        
        for item in layer2_data['original_items']:
            orig_idx = item['original_index']
            curr_idx = item['current_index']
            
            original_index_lookup[orig_idx] = {
                'layer3_cluster': layer3_label,
                'layer2_cluster': layer2_label,
                'current_index': curr_idx,
                'failed_summary': item['failed_summary']
            }
            
            current_index_lookup[curr_idx] = {
                'layer3_cluster': layer3_label,
                'layer2_cluster': layer2_label,
                'original_index': orig_idx,
                'failed_summary': item['failed_summary']
            }

# Save lookup tables
original_lookup_file = os.path.join(OUTPUT_DIR, "original_index_lookup.json")
current_lookup_file = os.path.join(OUTPUT_DIR, "current_index_lookup.json")

with open(original_lookup_file, 'w', encoding='utf-8') as f:
    json.dump(original_index_lookup, f, indent=2, ensure_ascii=False)

with open(current_lookup_file, 'w', encoding='utf-8') as f:
    json.dump(current_index_lookup, f, indent=2, ensure_ascii=False)

print(f"✓ Created lookup tables:")
print(f"  - Original index lookup: {original_lookup_file} ({len(original_index_lookup)} entries)")
print(f"  - Current index lookup: {current_lookup_file} ({len(current_index_lookup)} entries)")

# Show example
sample_orig_idx = list(original_index_lookup.keys())[0]
print(f"\nExample lookup by original_index {sample_orig_idx}:")
print(f"  {original_index_lookup[sample_orig_idx]}")

✓ Created lookup tables:
  - Original index lookup: results/connecting_layers/original_index_lookup.json (645 entries)
  - Current index lookup: results/connecting_layers/current_index_lookup.json (645 entries)

Example lookup by original_index 95:
  {'layer3_cluster': 0, 'layer2_cluster': 29, 'current_index': 34, 'failed_summary': 'The model likely focused on Mark’s later admission, “I’ve actually been worrying if we made the right decision all along,” and inferred that his underlying emotional state is anxiety rather than genuine excitement, so it chose “No.” In contrast, the human answer interprets people as often feeling mixed emotions, so Mark can still be truly excited about the new house while also worrying about the decision. Because the text supports both a conflicted-but-excited reading and a worry-dominated reading, both answers are plausible and the model’s response is also reasonable.'}


## Statistics

In [9]:
print("="*70)
print("HIERARCHY STATISTICS")
print("="*70)
print(f"Layer 3 clusters: {len(complete_hierarchy)}")
print(f"Layer 2 clusters: {len(layer2_clusters)}")
print(f"Total original items: {len(original_index_lookup)}")
print()
print(f"Layer 3 cluster sizes (number of layer 2 clusters):")
for cluster_data in complete_hierarchy:
    print(f"  Cluster {cluster_data['layer3_cluster_label']}: "
          f"{cluster_data['layer3_cluster_size']} layer 2 clusters, "
          f"{cluster_data['total_original_items']} original items")
print("="*70)

HIERARCHY STATISTICS
Layer 3 clusters: 10
Layer 2 clusters: 80
Total original items: 645

Layer 3 cluster sizes (number of layer 2 clusters):
  Cluster 0: 2 layer 2 clusters, 17 original items
  Cluster 1: 8 layer 2 clusters, 69 original items
  Cluster 2: 16 layer 2 clusters, 174 original items
  Cluster 3: 10 layer 2 clusters, 100 original items
  Cluster 4: 13 layer 2 clusters, 62 original items
  Cluster 5: 8 layer 2 clusters, 49 original items
  Cluster 6: 12 layer 2 clusters, 88 original items
  Cluster 7: 5 layer 2 clusters, 72 original items
  Cluster 8: 1 layer 2 clusters, 2 original items
  Cluster 9: 5 layer 2 clusters, 12 original items
